In [165]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [167]:
titanic = pd.read_csv("Titanic.csv")

In [169]:
titanic.head()

,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [171]:
titanic.shape

(891, 12)

In [173]:
titanic.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Gender           0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [175]:
titanic['Embarked'].unique

<bound method Series.unique of 0      S
1      C
2      S
3      S
4      S
      ..
886    S
887    S
888    S
889    C
890    Q
Name: Embarked, Length: 891, dtype: object>

In [177]:
titanic['Family_Size'] = titanic['SibSp'] + titanic['Parch'] + 1
titanic['Gender'] = titanic['Gender'].map({'male':0, 'female':1}) 
titanic = titanic.dropna()
titanic['Embarked'] = titanic['Embarked'].map({'C':0, 'S':1, 'Q':2})

In [179]:
titanic.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Gender         0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
Family_Size    0
dtype: int64

In [181]:
y = titanic['Survived']
x = titanic[['Age','Gender', 'Family_Size', 'Embarked', 'Fare']]

### Train test split

In [183]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.20)

### GridSearch for SVM

In [185]:
from sklearn.model_selection import GridSearchCV

parameters = {'kernel':['linear', 'rbf', 'poly'], 'C':[1, 5, 10]}

svc = SVC()

clf = GridSearchCV(svc, param_grid = parameters, cv = 3, verbose=True, n_jobs=-1)

final_clf = clf.fit(x_train, y_train)

print(sorted(final_clf.cv_results_.keys()))

Fitting 3 folds for each of 9 candidates, totalling 27 fits
['mean_fit_time', 'mean_score_time', 'mean_test_score', 'param_C', 'param_kernel', 'params', 'rank_test_score', 'split0_test_score', 'split1_test_score', 'split2_test_score', 'std_fit_time', 'std_score_time', 'std_test_score']


In [186]:
print(final_clf.best_estimator_)
print(final_clf.best_params_)

SVC(C=1, kernel='linear')
{'C': 1, 'kernel': 'linear'}


### SVC model from GridSearch best parameters

In [189]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

from sklearn import metrics

svc = SVC(kernel = 'linear', C=1)
svc.fit(x_train, y_train) # fiting the data to the model

y_pred = svc.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

print("Accuracy:", metrics.accuracy_score(y_test, y_pred))

[[ 8  3]
 [ 8 18]]
              precision    recall  f1-score   support

           0       0.50      0.73      0.59        11
           1       0.86      0.69      0.77        26

    accuracy                           0.70        37
   macro avg       0.68      0.71      0.68        37
weighted avg       0.75      0.70      0.71        37

Accuracy: 0.7027027027027027


### GridSearch for Random Forest Classifier

In [204]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

x, y = make_classification(n_samples=200, n_features=10, n_classes=2, random_state=42)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


parameters = {'n_estimators': [10, 50, 100, 200]}

rf = RandomForestClassifier(random_state=42)

clf = GridSearchCV(estimator=rf, param_grid=parameters, cv=5, scoring='accuracy', n_jobs=-1)

final_clf = clf.fit(x_train, y_train)

print(sorted(final_clf.cv_results_.keys()))

['mean_fit_time', 'mean_score_time', 'mean_test_score', 'param_n_estimators', 'params', 'rank_test_score', 'split0_test_score', 'split1_test_score', 'split2_test_score', 'split3_test_score', 'split4_test_score', 'std_fit_time', 'std_score_time', 'std_test_score']


In [206]:
print(final_clf.best_estimator_)
print(final_clf.best_params_)

RandomForestClassifier(random_state=42)
{'n_estimators': 100}


### Random Forest model from GridSearch best parameters

In [210]:
rfc = RandomForestClassifier(n_estimators=100, random_state=42)

rfc.fit(x_train, y_train)

rfc_predict = rfc.predict(x_test) 

from sklearn.metrics import confusion_matrix
cm_rf = confusion_matrix(y_test, rfc_predict)

print(cm_rf)
print(classification_report(y_test, rfc_predict))
print("Accuracy is:", metrics.accuracy_score(y_test,rfc_predict)*100) 

[[15  1]
 [ 3 21]]
              precision    recall  f1-score   support

           0       0.83      0.94      0.88        16
           1       0.95      0.88      0.91        24

    accuracy                           0.90        40
   macro avg       0.89      0.91      0.90        40
weighted avg       0.91      0.90      0.90        40

Accuracy is: 90.0


### The difference between RandomSearchCV and GridSearchCV.
#### GridSearchCV uses all the hyperparameters that are specified; RandomSearchCV randomly selects from a fixed set of hyperparameters that have been predefined.  Therefore, RandomSearchCV will likely be faster, as GridSearchCV has to perform more work for potentially more specified parameters. 